In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Conv2D,
    Flatten,
    Dropout,
    MaxPooling2D,
    Activation,
    BatchNormalization,
    GlobalAveragePooling2D,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras.metrics import AUC
from sklearn.model_selection import train_test_split


def load_data(
    csv_path="../input/train.csv",
    img_dir="../input/train/",
    batch_size=32,
    mode="categorical",
):
    """
    Load training and validation generators from the CSV file.
    """
    df = pd.read_csv(csv_path)
    df["has_cactus"] = df["has_cactus"].astype(str)

    train_df, val_df = train_test_split(
        df, test_size=0.1, stratify=df["has_cactus"], random_state=42
    )

    gen = ImageDataGenerator(
        rescale=1.0 / 255.0,
        horizontal_flip=True,
        vertical_flip=True,
    )

    train_gen = gen.flow_from_dataframe(
        train_df,
        directory=img_dir,
        x_col="id",
        y_col="has_cactus",
        has_ext=True,
        target_size=(32, 32),
        class_mode=mode,
        batch_size=batch_size,
        shuffle=True,
    )

    val_gen = gen.flow_from_dataframe(
        val_df,
        directory=img_dir,
        x_col="id",
        y_col="has_cactus",
        has_ext=True,
        target_size=(32, 32),
        class_mode=mode,
        batch_size=batch_size,
        shuffle=False,
    )

    return train_gen, val_gen




2026-05-07 08:29:10.152277: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778142550.166870      56 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778142550.171237      56 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-07 08:29:10.189161: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# Create generators
trainGen, valGen = load_data(batch_size=32)




Found 12757 validated image filenames belonging to 2 classes.
Found 1418 validated image filenames belonging to 2 classes.


In [3]:
def build_model():
    model = Sequential()
    model.add(
        Conv2D(
            32,
            (3, 3),
            padding="same",
            kernel_regularizer=l2(1e-4),
            input_shape=(32, 32, 3),
        )
    )
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(Conv2D(32, (3, 3), kernel_regularizer=l2(1e-4)))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.2))

    model.add(Conv2D(64, (3, 3), padding="same", kernel_regularizer=l2(1e-4)))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(Conv2D(64, (3, 3), kernel_regularizer=l2(1e-4)))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    model.add(Conv2D(128, (3, 3), padding="same", kernel_regularizer=l2(1e-4)))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(Conv2D(128, (3, 3), kernel_regularizer=l2(1e-4)))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(BatchNormalization())

    model.add(GlobalAveragePooling2D())
    model.add(Dense(2))
    model.add(BatchNormalization())
    model.add(Activation("softmax"))
    return model


model = build_model()
opt = Adam(learning_rate=1e-4)
model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=[AUC(name="auc")])

cbs = [
    ReduceLROnPlateau(monitor="loss", factor=0.5, patience=1, min_lr=1e-5, verbose=1)
]

steps_per_epoch = trainGen.samples // trainGen.batch_size
validation_steps = valGen.samples // valGen.batch_size

model.fit(
    trainGen,
    steps_per_epoch=steps_per_epoch,
    epochs=50,  # increased epochs for better AUC
    validation_data=valGen,
    validation_steps=validation_steps,
    callbacks=cbs,
    shuffle=True,
)




/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-05-07 08:29:18.635387: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778142558.635859      56 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:06:00.0, compute capability: 8.6
/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` ca

Epoch 1/50


I0000 00:00:1778142562.220380     107 service.cc:148] XLA service 0x7f77b00035d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778142562.220448     107 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-07 08:29:22.302730: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778142562.646087     107 cuda_dnn.cc:529] Loaded cuDNN version 90300


 11/398 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - auc: 0.7607 - loss: 0.6331

I0000 00:00:1778142566.486524     107 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


398/398 ━━━━━━━━━━━━━━━━━━━━ 19s 30ms/step - auc: 0.9185 - loss: 0.4223 - val_auc: 0.6274 - val_loss: 0.8157 - learning_rate: 1.0000e-04
Epoch 2/50
  1/398 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.9951 - loss: 0.2703

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:107: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


398/398 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - auc: 0.9951 - loss: 0.2703 - val_auc: 0.6386 - val_loss: 0.8003 - learning_rate: 1.0000e-04
Epoch 3/50
396/398 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - auc: 0.9701 - loss: 0.3274
Epoch 3: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
398/398 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - auc: 0.9701 - loss: 0.3273 - val_auc: 0.9079 - val_loss: 0.4493 - learning_rate: 1.0000e-04
Epoch 4/50
  1/398 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.9990 - loss: 0.2822
Epoch 4: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.
398/398 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - auc: 0.9990 - loss: 0.2822 - val_auc: 0.9149 - val_loss: 0.4421 - learning_rate: 5.0000e-05
Epoch 5/50
398/398 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - auc: 0.9781 - loss: 0.2932
Epoch 5: ReduceLROnPlateau reducing learning rate to 1.249999968422344e-05.
398/398 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - auc: 0.9782 - loss: 0.2932 - val_auc: 0.9626 - val_loss: 0.3283 - learning_ra

In [4]:
test_csv_path = "../input/sample_submission.csv"
test_img_dir = "../input/test/"
test_df = pd.read_csv(test_csv_path)

preds = np.empty((len(test_df),), dtype=np.float32)

for i in tqdm(range(len(test_df)), desc="Predicting"):
    img_path = os.path.join(test_img_dir, test_df.loc[i, "id"])
    img = Image.open(img_path).convert("RGB")
    arr = np.array(img).astype(np.float32) / 255.0
    prob = model.predict(arr.reshape(1, 32, 32, 3), verbose=0)[0][1]
    preds[i] = prob

test_df["has_cactus"] = preds
submission_path = "sample_submission.csv"
test_df.to_csv(submission_path, index=False)

Predicting: 100%|██████████| 3325/3325 [02:18<00:00, 24.03it/s]
